In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import tifffile
from skimage.measure import regionprops_table

# ----------------------- CONFIG -----------------------
folders_to_read = [
    Path(r"Z:\Jorge\SPACEFISH_analysis\2026\v115-3Drep2-0101-indinuc"),
    Path(r"Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-0101-indinuc"),
]
subfolders_to_skip = [
    Path(r"Z:\Jorge\SPACEFISH_analysis\2026\v115-3Drep2-0101-indinuc\results\dev1_2_6h"),
]
output_dir = Path(r"c:\Users\taylorhearn\git_repos\image_quantification\New_Spacefish")
# ------------------------------------------------------

output_dir.mkdir(parents=True, exist_ok=True)
skip_resolved = {p.resolve() for p in subfolders_to_skip}


def is_skipped(folder: Path) -> bool:
    rp = folder.resolve()
    return rp in skip_resolved or any(parent in skip_resolved for parent in rp.parents)


def find_image_folders(roots):
    """An 'image folder' is any folder containing a *count_table*.csv."""
    found = []
    for root in roots:
        for csv in root.rglob("*count_table*.csv"):
            folder = csv.parent
            if is_skipped(folder):
                print(f"[SKIP - excluded] {folder}")
                continue
            found.append(folder)
    return sorted(set(found))


def centroids_from_mask(mask: np.ndarray):
    """Return (props_df, (Xmax, Ymax, Zmax)). Mask assumed (Z, Y, X) if 3D."""
    mask = np.squeeze(mask)
    props = pd.DataFrame(regionprops_table(mask, properties=("label", "centroid")))
    if mask.ndim == 3:
        props = props.rename(columns={
            "centroid-0": "nucleus_centroid_z",
            "centroid-1": "nucleus_centroid_y",
            "centroid-2": "nucleus_centroid_x",
        })
        zmax, ymax, xmax = mask.shape
    else:  # 2D fallback
        props = props.rename(columns={
            "centroid-0": "nucleus_centroid_y",
            "centroid-1": "nucleus_centroid_x",
        })
        props["nucleus_centroid_z"] = np.nan
        ymax, xmax = mask.shape
        zmax = 1
    return props, (int(xmax), int(ymax), int(zmax))



In [ ]:

all_tables = []
image_folders = find_image_folders(folders_to_read)
print(f"Found {len(image_folders)} candidate image folders.\n")

for folder in image_folders:
    count_csv = next(iter(folder.glob("*count_table*.csv")), None)
    seg_path = next(iter(folder.glob("*seg_mask*.tif")), None)

    if seg_path is None:
        print(f"[SKIP - no seg_mask] {folder}")
        continue

    df = pd.read_csv(count_csv)

    mask = tifffile.imread(str(seg_path))
    props, (xmax, ymax, zmax) = centroids_from_mask(mask)

    # Merge centroids onto the count table by nucleus label.
    df = df.merge(
        props.rename(columns={"label": "nucleus"}),
        on="nucleus", how="left",
    )

    # Fallback: fill any missing centroids from nucleus_xyz.csv (ID, x, y, z).
    xyz_path = folder / "nucleus_xyz.csv"
    if xyz_path.exists() and df[["nucleus_centroid_x",
                                 "nucleus_centroid_y",
                                 "nucleus_centroid_z"]].isna().any().any():
        xyz = pd.read_csv(xyz_path).set_index("ID")
        for axis in ("x", "y", "z"):
            col = f"nucleus_centroid_{axis}"
            missing = df[col].isna()
            df.loc[missing, col] = df.loc[missing, "nucleus"].map(xyz[axis])

    df["Xmax"] = xmax
    df["Ymax"] = ymax
    df["Zmax"] = zmax

    # Write a per-image augmented copy into New_Spacefish (originals untouched).
    out_name = f"{folder.name}_count_table_xyz.csv"
    df.to_csv(output_dir / out_name, index=False)
    all_tables.append(df)
    print(f"[OK] {folder.name}: {len(df)} nuclei, extent (X,Y,Z)=({xmax},{ymax},{zmax})")

# Concatenate every nucleus from every image into one table.
if all_tables:
    combined = pd.concat(all_tables, ignore_index=True)
    combined.to_csv(output_dir / "all_nuclei.csv", index=False)
    print(f"\nCombined table: {combined.shape[0]} nuclei x {combined.shape[1]} columns")
    print(f"Saved -> {output_dir / 'all_nuclei.csv'}")
else:
    print("\nNo folders produced output (no seg_mask found).")

[SKIP - excluded] Z:\Jorge\SPACEFISH_analysis\2026\v115-3Drep2-0101-indinuc\results\dev1_2_6h
Found 24 candidate image folders.

[OK] dev1_3_6h: 1000 nuclei, extent (X,Y,Z)=(3933,3945,126)
[OK] dev1_4_6h: 2142 nuclei, extent (X,Y,Z)=(3942,3972,126)
[OK] dev3_2_day1: 944 nuclei, extent (X,Y,Z)=(4059,4074,126)
[OK] dev3_3_day1: 1032 nuclei, extent (X,Y,Z)=(3945,3942,126)
[OK] dev6_2_day2: 862 nuclei, extent (X,Y,Z)=(3930,3978,126)


In [ ]:
combined.head()

NameError: name 'image_folders' is not defined